# Bộ lọc SMS: chặn nhầm cũng có giá
**Sau bài 03 · 3–4 giờ, làm cá nhân hoặc cặp.** Đây là thời gian dự án bổ sung, không cộng vào 150 phút của bài học.

**Sản phẩm:** Một pipeline kNN, ngưỡng chọn bằng validation, hàm dự đoán tin nhắn mới và báo cáo lỗi.

Đọc → dự đoán → chạy mã mẫu → tự điền TODO → kiểm thử → giải thích. Các TODO cố ý dừng bằng `NotImplementedError`; đây không phải lỗi môi trường. Không thay kiểm thử bằng giá trị viết cứng. Giữ seed, Restart Kernel và chạy lại trước khi nộp.

Nguồn, điều kiện sử dụng và giới hạn dữ liệu: [bảng nguồn](../../course/DATA_SOURCES.md). Không tải mô hình pickle/joblib không rõ nguồn.


In [ ]:
from pathlib import Path
import sys, json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p/'mlcourse').is_dir()), None)
assert ROOT is not None, 'Mở notebook trong bản tải đầy đủ của repository.'
if str(ROOT) not in sys.path: sys.path.insert(0, str(ROOT))
from threadpoolctl import threadpool_limits
_threads = threadpool_limits(limits=2)
SEED = 42

OUT = ROOT/'outputs/projects/02_sms_filter'
OUT.mkdir(parents=True, exist_ok=True)


## 1. Chốt quyết định trước khi nhìn test
Giả định minh họa: chặn nhầm tin thường tốn 5 đơn vị; bỏ sót spam tốn 1. Đây là chi phí do dự án đặt ra, không phải số liệu kinh doanh đo được.
$$C=5FP+FN.$$
Chỉ dùng hai đặc trưng hình thức, không hiểu ngữ nghĩa SMS. Tập dữ liệu chủ yếu tiếng Anh; không suy rộng sang SMS tiếng Việt.


In [ ]:
from mlcourse.sms import load_sms, FEATURES, text_features
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import confusion_matrix, classification_report
import joblib
d, tr, va, te, raw_count = load_sms(ROOT/'course/bai02/data', SEED)
assert not (set(tr)&set(va) or set(tr)&set(te) or set(va)&set(te))
assert d.normal.is_unique
display(d.groupby('split').y.agg(['count','mean']))
print('Số dòng gốc / sau khử trùng:',raw_count,len(d))


## 2. TODO: chi phí sai quyết định
`y` là nhãn 0/1, `p` là score lớp spam, `threshold` trong [0,1]. Dự đoán spam khi `p >= threshold`. Trả tổng chi phí, không chia số mẫu.


In [ ]:
def decision_cost(y, p, threshold):
    y, p = np.asarray(y), np.asarray(p)
    # TODO: đếm FP và FN, tính 5*FP+FN.
    raise NotImplementedError('Hoàn thiện decision_cost')


Gợi ý: FP là nhãn thật 0 nhưng dự đoán 1. Kiểm thử nhỏ: y=[0,0,1,1], score=[.9,.1,.8,.2], ngưỡng .5 tạo một FP và một FN.


In [ ]:
assert decision_cost([0,0,1,1],[.9,.1,.8,.2],.5)==6
assert decision_cost([0,1],[.49,.5],.5)==0
assert decision_cost([0,1],[.1,.9],1.0)==1


## 3. Mã mẫu: học trên train, khảo sát validation
Scaler chỉ fit trên train. Giữ k=5 để tập trung vào ngưỡng; score kNN là tỷ lệ láng giềng spam, không mặc nhiên được hiệu chỉnh xác suất.


In [ ]:
model = make_pipeline(StandardScaler(), KNeighborsClassifier(n_neighbors=5))
model.fit(d.loc[tr, FEATURES], d.loc[tr,'y'])
pv = model.predict_proba(d.loc[va,FEATURES])[:,1]
thresholds = np.linspace(0,1,21)
curve = pd.DataFrame({'threshold':thresholds,'cost':[decision_cost(d.loc[va,'y'],pv,t) for t in thresholds]})
curve.plot(x='threshold',y='cost',marker='o',legend=False)
plt.ylabel('Validation cost'); plt.savefig(OUT/'threshold.png'); plt.show()


## 4. TODO: khóa ngưỡng trước khi xem test
Chọn ngưỡng chi phí validation nhỏ nhất; nếu hòa, lấy ngưỡng lớn hơn (quy tắc đặt trước).


In [ ]:
def choose_threshold(curve):
    # TODO: sắp xếp cost tăng, threshold giảm; trả float.
    raise NotImplementedError('Chọn ngưỡng từ validation')


In [ ]:
assert choose_threshold(pd.DataFrame({'threshold':[.2,.4,.6],'cost':[2,1,1]}))==.6
threshold = choose_threshold(curve)
print('Ngưỡng đã khóa:',threshold)
assert 0<=threshold<=1


## 5. Mở test một lần — mã mẫu
Báo cáo cả baseline luôn ham và ngưỡng 0.5. Nếu kết quả chưa tốt, ghi nhận; không quay lại chọn lại ngưỡng theo test.


In [ ]:
pt = model.predict_proba(d.loc[te,FEATURES])[:,1]
yt = d.loc[te,'y'].to_numpy()
pred = (pt>=threshold).astype(int)
print(classification_report(yt,pred,zero_division=0))
cm = confusion_matrix(yt,pred,labels=[0,1]); print(cm)
report = {'threshold':threshold,'test_cost':decision_cost(yt,pt,threshold),'default_cost':decision_cost(yt,pt,.5),'always_ham_cost':int(yt.sum()),'test_count':len(yt),'confusion_matrix':cm.tolist(),'seed':SEED}
display(pd.Series(report))
errors = d.loc[te,['message_id','label']].copy()
errors['prediction']=pred; errors['spam_score']=pt
errors.to_csv(OUT/'predictions.csv',index=False)
(OUT/'metrics.json').write_text(json.dumps(report,indent=2),encoding='utf-8')
joblib.dump({'model':model,'threshold':threshold,'features':FEATURES},OUT/'sms_model.joblib')


## 6. Dự đoán dữ liệu mới và giải thích lỗi
Chỉ nạp artifact do chính bạn tạo. Không coi nhãn spam là bằng chứng lừa đảo. Chọn hai false positive và hai false negative nếu có, đọc tại máy và giải thích giới hạn của hai đặc trưng.


In [ ]:
bundle = joblib.load(OUT/'sms_model.joblib')
def predict_sms(text):
    row = pd.DataFrame([text_features(text)])[bundle['features']]
    score = float(bundle['model'].predict_proba(row)[0,1])
    return {'spam_score':score,'flag_spam':score>=bundle['threshold']}
print(predict_sms('Hi, see you at the library tomorrow.'))
print(predict_sms('WIN a prize now! Call 123456789.'))
assert predict_sms(d.loc[te[0],'text'])['flag_spam']==bool(pred[0])


## Hoàn thiện sản phẩm và tự kiểm
Baseline hai đặc trưng có giới hạn rõ rệt. Mở rộng sau Bài 5: thử TF–IDF + logistic với vocabulary chỉ fit train, chọn trên validation và dùng một test mới nếu đã dùng test cũ để điều chỉnh thiết kế.

Nộp notebook đã chạy, thư mục đầu ra của dự án và `REPORT.md` tự viết: vấn đề → dữ liệu → phương pháp → kết quả → giới hạn. Dẫn nguồn, ghi seed/cách chia và mô tả ít nhất một trường hợp thất bại. Không chỉ chụp hình điểm số.

**Câu hỏi phản tư:** Điều gì trong kết quả là bằng chứng, điều gì mới là giả thuyết? Nếu dữ liệu mới khác dữ liệu học, bạn cần kiểm tra gì?

**Nhiệm vụ mở:** thay đổi một lựa chọn có lý do, xác định tiêu chí trước khi chạy và so sánh trên validation (hoặc phân tích độ ổn định với bài phân cụm). Không liên tục xem test để chọn cách làm.
